# load dataset

In [1]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train.head()
train.shape

(8693, 14)

# Exploratory Data Analysis (EDA)

In [2]:
train.info()
train.describe()
train.isnull().sum()
train["Transported"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


Transported
True     4378
False    4315
Name: count, dtype: int64

# Preprocessing

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

target = "Transported"
X = train.drop(columns=[target])
y = train[target]

cat_features = X.select_dtypes(include=["object"]).columns
num_features = X.select_dtypes(include=["int64", "float64"]).columns

cat_transformer = OneHotEncoder(handle_unknown="ignore")
num_transformer = Pipeline([
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", cat_transformer, cat_features),
    ("num", num_transformer, num_features)
])

# Train Classification Model

In [4]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    ))
])

model.fit(X, y)

C:\Users\Asus TUF\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:199: UserWarning: [11:41:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,steps,"[('preprocess', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Evaluate Model Performance

In [5]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X)

print("Training Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))

Training Accuracy: 0.8788680547567008
              precision    recall  f1-score   support

       False       0.90      0.85      0.87      4315
        True       0.86      0.91      0.88      4378

    accuracy                           0.88      8693
   macro avg       0.88      0.88      0.88      8693
weighted avg       0.88      0.88      0.88      8693



# Improve Model (Hyperparameter Tuning)

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [4, 6, 8],
    "classifier__learning_rate": [0.03, 0.05, 0.1]
}

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X, y)

best_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

C:\Users\Asus TUF\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:199: UserWarning: [11:43:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 6, 'classifier__n_estimators': 300}
Best CV Score: 0.7962741415863835


# Create Submission File

In [7]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": best_model.predict(test)
})

submission.to_csv("submission.csv", index=False)